# Módulo modelado: conjunto, variables y modelos
Lo usan los notebooks 19 a 23 con `%run -i modulos/modelado.ipynb` (después de `modulos/comun.ipynb`).

- **Pregunta:** ¿cuánto va a tardar un caso? Se responde para el **grupo** del caso: un tipo de proceso no penal en un
  territorio y una materia (el Anuario no publica causas una por una).
- **Objetivo:** el **tiempo estimado de resolución** del notebook 10, `pendientes al cierre ÷ resueltas en el año × 365`
  días: cuánto tardarían los juzgados en terminar la fila que quedó si siguen al mismo ritmo. Es un número, así que el
  problema es de **regresión**. El modelo aprende su logaritmo, `log(1 + días)`, porque los tiempos van de 0 a miles de
  días y en escala lineal unos pocos procesos lentísimos dominarían el aprendizaje.
- **Conjunto:** los procesos no penales con **al menos 10 causas atendidas** en 2023 (con menos, el «tiempo» sale de una o
  dos causas y es una anécdota, no una medida) y con al menos una resuelta (si no se resolvió nada, el tiempo no existe).
  El primer filtro usa un dato de entrada (stock heredado + ingresos), no la respuesta.
- **Variables:** solo lo que se conoce **al presentar el caso**: stock heredado, causas que entran, materia, territorio,
  juzgados, personal y la **historia** de la materia (cómo cambiaron sus resoluciones el año anterior, capítulo 9).
  Todo lo que depende de las resueltas o del stock final del año queda fuera: es la respuesta disfrazada (*fuga*).

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (AdaBoostRegressor, BaggingRegressor, GradientBoostingRegressor,
                              RandomForestRegressor, StackingRegressor)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

SEMILLA = 42
MODELO_DIR = CURATED / "modelo"
MODELO_DIR.mkdir(parents=True, exist_ok=True)
IMG = RAIZ / "web" / "img"
OBJETIVO = "log_tiempo"            # log(1 + días): lo que aprende el modelo
DIAS = "tiempo_dias"               # el tiempo en días, para leer los resultados
TOPE_DIAS = 3650                   # 10 años: por encima, «más de 10 años» (13 procesos)
MIN_CAUSAS = 10                    # causas atendidas mínimas para que el tiempo sea una medida
CV5 = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)
ANIO = 2023

# Columnas que identifican la fila; no entran al modelo.
IDENTIFICADORES = ["cuadro_origen", "pagina_pdf", "tipo_proceso", "territorio"]

CATEGORICAS = ["materia_homologada", "ambito", "departamento_derivado"]
NUMERICAS_ORIGINALES = [
    "pendientes_inicio_efectivo",       # stock heredado de 2022 (en civil, las readecuadas a la Ley 439)
    "nuevas_ingresadas",
    "recibidas_excusa_recusacion",
    "preliminares_formalizados",        # solo civil; en otras materias esa forma de ingreso no existe -> 0
    "cautelares_formalizados",          # idem
    "ingresos_totales",                 # todas las formas de ingreso del año
    "atendidas",                        # stock heredado + ingresos
    "num_juzgados_pagina",              # juzgados de esa materia en ese territorio
    "recurso_juzgados_publicados",
    "recurso_tribunales_publicados",
    "recurso_conciliadores_publicados",
    "personal_items_mujer",
    "personal_items_varon",
    "personal_items_acefalias",         # puestos vacantes
    "personal_items_total",
    "personal_remun_total",
]
CEROS_ESTRUCTURALES = ["preliminares_formalizados", "cautelares_formalizados"]
ORIGINALES = CATEGORICAS + NUMERICAS_ORIGINALES

# Variables nuevas -> columnas originales que necesitan (el despliegue las recalcula con la misma fórmula).
DEPENDENCIAS = {
    "log_stock_inicial": ["pendientes_inicio_efectivo"],
    "log_ingresos": ["ingresos_totales"],
    "presion_stock": ["pendientes_inicio_efectivo", "ingresos_totales"],
    "ingresos_por_juzgado": ["ingresos_totales", "num_juzgados_pagina"],
    "stock_por_juzgado": ["pendientes_inicio_efectivo", "num_juzgados_pagina"],
    "prop_acefalias": ["personal_items_acefalias", "personal_items_total"],
    "prop_excusa": ["recibidas_excusa_recusacion", "ingresos_totales"],
    "sin_stock_inicial": ["pendientes_inicio_efectivo"],
}
FORMULAS = {
    "log_stock_inicial": "log(1 + stock heredado)",
    "log_ingresos": "log(1 + ingresos del año)",
    "presion_stock": "stock heredado / (ingresos del año + 1)",
    "ingresos_por_juzgado": "ingresos del año / juzgados",
    "stock_por_juzgado": "stock heredado / juzgados",
    "prop_acefalias": "puestos vacantes / puestos totales",
    "prop_excusa": "recibidas por excusa / (ingresos del año + 1)",
    "sin_stock_inicial": "1 si no había stock heredado",
    "tendencia_resueltas_materia": "resueltas del año anterior / resueltas de dos años antes - 1 (materia y ámbito)",
}
INGENIERIA = list(DEPENDENCIAS)
# Variable histórica: sale de la serie por materia del capítulo 9 (2020-2023), no de la fila.
HISTORICAS = ["tendencia_resueltas_materia"]
COMPLETA = ORIGINALES + INGENIERIA + HISTORICAS

# Columnas excluidas por fuga de información: se calculan con las resueltas o con el stock final del año.
FUGA = {
    "resueltas": "es parte de la respuesta: el tiempo se divide por las resueltas",
    "pendientes_fin": "es parte de la respuesta: el tiempo es pendientes al cierre / resueltas",
    "duracion_estimada_dias": "es el objetivo",
    "duracion_estimada_winsorizada": "es el objetivo recortado",
    "tasa_resolucion": "resueltas / ingresos",
    "tasa_resolucion_con_readecuadas": "usa resueltas",
    "tasa_resolucion_solo_nuevas": "usa resueltas",
    "tasa_congestion": "atendidas / resueltas",
    "tasa_pendencia": "usa pendientes al cierre",
    "acumulacion_neta_anual": "ingresos - resueltas",
    "flag_acumula_mora": "usa resueltas",
    "flag_sin_resolucion_anual": "usa resueltas",
    "flag_sin_movimiento": "usa resueltas",
    "estado_gestion": "clasifica la fila según sus resueltas",
    "tasa_congestion_winsorizada": "usa resueltas",
    "log_resueltas": "usa resueltas",
    "log_pendientes_fin": "usa pendientes al cierre",
    "es_outlier_congestion_iqr": "marca construida con la tasa de congestión",
    "es_outlier_duracion_iqr": "marca construida con el tiempo",
}
NOMBRES = {
    "materia_homologada": "materia",
    "ambito": "capital o provincia",
    "departamento_derivado": "departamento",
    "pendientes_inicio_efectivo": "stock heredado",
    "nuevas_ingresadas": "causas nuevas",
    "recibidas_excusa_recusacion": "recibidas por excusa",
    "preliminares_formalizados": "preliminares formalizados",
    "cautelares_formalizados": "cautelares formalizados",
    "ingresos_totales": "ingresos del año",
    "atendidas": "causas atendidas",
    "num_juzgados_pagina": "juzgados (materia y territorio)",
    "recurso_juzgados_publicados": "juzgados del territorio",
    "recurso_tribunales_publicados": "tribunales del territorio",
    "recurso_conciliadores_publicados": "conciliadores del territorio",
    "personal_items_mujer": "puestos ocupados por mujeres",
    "personal_items_varon": "puestos ocupados por varones",
    "personal_items_acefalias": "puestos vacantes",
    "personal_items_total": "puestos totales",
    "personal_remun_total": "remuneración total",
    "log_stock_inicial": "log del stock heredado",
    "log_ingresos": "log de los ingresos",
    "presion_stock": "presión del stock",
    "ingresos_por_juzgado": "ingresos por juzgado",
    "stock_por_juzgado": "stock por juzgado",
    "prop_acefalias": "proporción de vacantes",
    "prop_excusa": "proporción por excusa",
    "sin_stock_inicial": "sin stock heredado",
    "tendencia_resueltas_materia": "tendencia histórica de la materia",
}

In [ ]:
def construir_variables(df):
    """Las 8 variables nuevas, fila por fila."""
    s = df["pendientes_inicio_efectivo"]
    i = df["ingresos_totales"]
    j = df["num_juzgados_pagina"]
    out = pd.DataFrame(index=df.index)
    out["log_stock_inicial"] = np.log1p(s)
    out["log_ingresos"] = np.log1p(i)
    out["presion_stock"] = s / (i + 1)
    out["ingresos_por_juzgado"] = i / j
    out["stock_por_juzgado"] = s / j
    out["prop_acefalias"] = df["personal_items_acefalias"] / df["personal_items_total"]
    out["prop_excusa"] = df["recibidas_excusa_recusacion"] / (i + 1)
    out["sin_stock_inicial"] = (s == 0).astype(float)
    return out


def resueltas_por_materia():
    """Causas resueltas por materia y ámbito (capital, provincia), 2019-2023: cuadros 9.1.2 y 9.1.6 del capítulo 9."""
    g = pd.read_parquet(PROCESSED / "causas_por_gestion.parquet")
    g = g[(g["tipo_fila_derivado"] == "dato") & g["ambito"].isin(["capital", "provincia"])]
    return g.pivot_table(index=["ambito", "materia_homologada"], columns="gestion", values="resueltas", aggfunc="sum")


def tendencia_historica(anio):
    """Para presentar un caso en `anio`: cómo cambiaron las resoluciones de su materia y ámbito el año anterior.
    resueltas(anio - 1) / resueltas(anio - 2) - 1. Para 2023 usa 2022 y 2021; para 2024, 2023 y 2022."""
    r = resueltas_por_materia()
    return (r[anio - 1] / r[anio - 2] - 1).rename("tendencia_resueltas_materia")


def agregar_historia(df, anio):
    t = tendencia_historica(anio)
    claves = list(zip(df["ambito"], df["materia_homologada"]))
    return pd.Series([t.get(k, np.nan) for k in claves], index=df.index, name="tendencia_resueltas_materia")


def tiempo_dias(pendientes_fin, resueltas):
    """pendientes al cierre / resueltas x 365, con tope de 10 años; NaN si no se resolvió nada."""
    r = resueltas.where(resueltas > 0)
    return (pendientes_fin / r * 365).clip(upper=TOPE_DIAS)


def cargar_procesos():
    """Todos los procesos no penales con movimiento (al menos una causa atendida), sin filtrar."""
    d = pd.read_parquet(CURATED / "features" / "dataset_analitico_curado.parquet")
    return d[(d["tipo_elemento_analitico"] == "proceso") & (d["atendidas"] > 0)].reset_index(drop=True)


def preparar(d, anio=ANIO):
    """Tipos convertidos, ceros estructurales, variables nuevas y la variable histórica."""
    datos = d[IDENTIFICADORES + ORIGINALES].copy()
    for c in NUMERICAS_ORIGINALES:
        datos[c] = datos[c].astype("float64")        # Int64 con <NA> -> float64 con NaN
    for c in CATEGORICAS:
        datos[c] = datos[c].astype("object")
    datos[CEROS_ESTRUCTURALES] = datos[CEROS_ESTRUCTURALES].fillna(0.0)
    datos = pd.concat([datos, construir_variables(datos)], axis=1)
    datos["tendencia_resueltas_materia"] = agregar_historia(datos, anio)
    return datos


def cargar_conjunto():
    """Filas del modelo: al menos MIN_CAUSAS atendidas y al menos una resuelta."""
    d = cargar_procesos()
    d = d[(d["atendidas"] >= MIN_CAUSAS) & (d["resueltas"] > 0)].reset_index(drop=True)
    datos = preparar(d)
    datos[DIAS] = tiempo_dias(d["pendientes_fin"].astype(float), d["resueltas"].astype(float))
    datos[OBJETIVO] = np.log1p(datos[DIAS])
    return datos


def particion(datos):
    """80 % entrenamiento, 20 % prueba; estratificada por quintil del tiempo para que ambas partes tengan
    procesos rápidos y lentos en la misma proporción."""
    quintil = pd.qcut(datos[OBJETIVO].rank(method="first"), 5, labels=False)
    return train_test_split(datos.index, test_size=0.2, stratify=quintil, random_state=SEMILLA)

In [ ]:
def preprocesador(columnas, escalar):
    """Imputación + escala (si el modelo la necesita) para números; imputación + one-hot para categorías."""
    num = [c for c in columnas if c not in CATEGORICAS]
    cat = [c for c in columnas if c in CATEGORICAS]
    pasos = [("imputar", SimpleImputer(strategy="median"))]
    if escalar:
        pasos.append(("escalar", StandardScaler()))
    transformadores = [("num", Pipeline(pasos), num)]
    if cat:
        transformadores.append(("cat", Pipeline([
            ("imputar", SimpleImputer(strategy="constant", fill_value="Desconocido")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat))
    return ColumnTransformer(transformadores)


class FormulaIngenua(BaseEstimator, RegressorMixin):
    """Referencia sin aprendizaje: si los juzgados resolvieran tanto como entra, la fila al cierre sería el stock
    heredado y tardaría stock heredado / ingresos x 365 días."""

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        dias = (X["pendientes_inicio_efectivo"] / (X["ingresos_totales"] + 1) * 365).clip(upper=TOPE_DIAS)
        return np.log1p(dias.to_numpy(dtype=float))


# nombre -> (familia, ¿necesita escala?)
FAMILIAS = {
    "Línea base (promedio)": ("referencia", False),
    "Fórmula ingenua": ("referencia", False),
    "Regresión lineal (Ridge)": ("referencia", True),
    "SVM": ("SVM", True),
    "Bagging": ("Bagging", False),
    "Random Forest": ("Bagging", False),
    "AdaBoost": ("Boosting", False),
    "Gradient Boosting": ("Boosting", False),
    "Stacking": ("Stacking", True),
}
MODELOS = list(FAMILIAS)
REFERENCIAS = ["Línea base (promedio)", "Fórmula ingenua"]
ENTRENABLES = [m for m in MODELOS if m not in REFERENCIAS]


def estimador(nombre, hp=None):
    """Estimador con sus hiperparámetros ajustados (hp: {modelo: {parámetro: valor}})."""
    hp = hp or {}
    p = dict(hp.get(nombre, {}))
    if nombre == "Línea base (promedio)":
        return DummyRegressor(strategy="mean")
    if nombre == "Regresión lineal (Ridge)":
        return Ridge(**p)
    if nombre == "SVM":
        return SVR(kernel="rbf", **p)
    if nombre == "Bagging":
        hoja = p.pop("estimator__min_samples_leaf", 1)
        return BaggingRegressor(estimator=DecisionTreeRegressor(min_samples_leaf=hoja, random_state=SEMILLA),
                                random_state=SEMILLA, n_jobs=-1, **p)
    if nombre == "Random Forest":
        return RandomForestRegressor(random_state=SEMILLA, n_jobs=-1, **p)
    if nombre == "AdaBoost":
        prof = p.pop("estimator__max_depth", 3)
        return AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=prof, random_state=SEMILLA),
                                 random_state=SEMILLA, **p)
    if nombre == "Gradient Boosting":
        return GradientBoostingRegressor(random_state=SEMILLA, **p)
    if nombre == "Stacking":
        base = [("svm", estimador("SVM", hp)), ("rf", estimador("Random Forest", hp)),
                ("gb", estimador("Gradient Boosting", hp))]
        return StackingRegressor(estimators=base, final_estimator=Ridge(alpha=1.0),
                                 cv=KFold(5, shuffle=True, random_state=SEMILLA), n_jobs=-1)
    raise KeyError(nombre)


def modelo(nombre, columnas, hp=None):
    """Pipeline completo: preprocesamiento + estimador. La fórmula ingenua no necesita preprocesamiento."""
    if nombre == "Fórmula ingenua":
        return FormulaIngenua()
    return Pipeline([("pre", preprocesador(columnas, FAMILIAS[nombre][1])), ("modelo", estimador(nombre, hp))])


def medir(y_real, y_pred):
    """Métricas en escala log(1 + días) y su lectura en días.
    - mae_log: error absoluto medio en log; exp(mae_log) es el factor por el que se equivoca «en promedio».
    - factor_mediano: la mitad de las estimaciones se equivoca por un factor menor que este (x1,5 = 50 % de más o de menos).
    - dentro_doble: proporción de procesos cuyo tiempo real quedó entre la mitad y el doble de lo estimado.
    - r2_log: qué parte de las diferencias entre procesos explica el modelo (0 = nada, 1 = todo).
    - error_dias_mediano: error típico en días."""
    y_real, y_pred = np.asarray(y_real, float), np.asarray(y_pred, float)
    e = np.abs(y_pred - y_real)
    return {
        "mae_log": e.mean(),
        "factor_mediano": float(np.exp(np.median(e))),
        "dentro_doble": float((e <= np.log(2)).mean()),
        "r2_log": 1 - ((y_pred - y_real) ** 2).sum() / ((y_real - y_real.mean()) ** 2).sum(),
        "error_dias_mediano": float(np.median(np.abs(np.expm1(y_pred) - np.expm1(y_real)))),
    }


ESCORE = "neg_mean_absolute_error"     # el ajuste minimiza el error absoluto medio en log


def importancia_permutacion_cv(nombre, columnas, X, y, hp, repeticiones=10):
    """Importancia por permutación medida en los pliegues de validación (nunca en la prueba):
    cuánto SUBE el error absoluto medio (en log) al barajar la variable."""
    medidas = []
    for tr, va in CV5.split(X, y):
        m = modelo(nombre, columnas, hp).fit(X.iloc[tr][columnas], y.iloc[tr])
        r = permutation_importance(m, X.iloc[va][columnas], y.iloc[va], scoring=ESCORE,
                                   n_repeats=repeticiones, random_state=SEMILLA, n_jobs=-1)
        medidas.append(pd.Series(r.importances_mean, index=columnas))
    tabla = pd.concat(medidas, axis=1)
    return (pd.DataFrame({"importancia": tabla.mean(axis=1), "desvio_entre_pliegues": tabla.std(axis=1)})
            .sort_values("importancia", ascending=False))


def guardar_figura(fig, nombre):
    fig.savefig(IMG / nombre, dpi=150, bbox_inches="tight")


def leer_hp():
    return json.loads((MODELO_DIR / "hiperparametros.json").read_text(encoding="utf-8"))


def meses(dias):
    """Texto corto para leer un tiempo: días si es menos de 2 meses, meses si es menos de 2 años, años si no."""
    if dias is None or (isinstance(dias, float) and np.isnan(dias)):
        return "sin dato"
    if dias < 60:
        return f"{dias:.0f} días"
    if dias < 730:
        return f"{dias / 30.44:.0f} meses"
    return f"{dias / 365.25:.1f} años".replace(".", ",")